# 🪐 Planet Hunters — starter notebook
**ML Challenge Hack4Hack 2026 · Round 1**

This notebook loads the data, plots a few stars, builds simple statistics for every star, trains a gradient-boosting model and writes a valid `submission.csv`.

It is a **baseline**: it ignores periods, dip shapes, centroids and flares, so it scores low. Beating it is your job. It runs in about 2 minutes on a laptop with 8 GB RAM.

Put this notebook in the same folder as the data files (or change `DATA` below).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DATA = '../data/raw'   # folder with the .npy and .csv files
train_flux = np.load(f'{DATA}/train_flux.npy', mmap_mode='r')        # (18000, 1315) electrons/s, NaN = gap
train_cen  = np.load(f'{DATA}/train_centroid.npy', mmap_mode='r')    # (18000, 2, 1315) millipixels
test_flux  = np.load(f'{DATA}/test_flux.npy', mmap_mode='r')
train = pd.read_csv(f'{DATA}/train_stars.csv')
test  = pd.read_csv(f'{DATA}/test_stars.csv')
cad   = pd.read_csv(f'{DATA}/train_cadences.csv')
t = cad.time_btjd.values
print(train_flux.shape, test_flux.shape)
train.label.value_counts()

## Look at the data
One star of each label: relative brightness (top) and centroid column offset (bottom). Orange ticks mark momentum dumps (`quality & 1`).

In [ ]:
labels = sorted(train.label.unique())
fig, axes = plt.subplots(len(labels), 1, figsize=(14, 2.2 * len(labels)), sharex=True)
dumps = t[(cad.quality.values & 1) > 0]
for ax, lab in zip(axes, labels):
    i = train.index[train.label == lab][0]
    f = train_flux[i] / np.nanmedian(train_flux[i]) - 1
    ax.plot(t, f * 1e3, '.', ms=2, color='k')
    for d in dumps: ax.axvline(d, color='orange', lw=0.5, alpha=0.6)
    ax.set_ylabel('ppt'); ax.set_title(f'{lab}  ({train.id[i]}, Tmag {train.tmag[i]:.1f})', fontsize=10, loc='left')
axes[-1].set_xlabel('time (BTJD)'); plt.tight_layout(); plt.show()

## Simple features
Normalise each star by its median, then compute spread, extremes, skewness and percentiles. Add the star properties from the CSV.

In [ ]:
def simple_features(flux, stars, batch=2000):
    out = []
    for s in range(0, len(flux), batch):
        f = np.asarray(flux[s:s + batch], np.float64)
        f = f / np.nanmedian(f, axis=1, keepdims=True) - 1
        out.append(pd.DataFrame({
            'std': np.nanstd(f, 1),
            'mad': np.nanmedian(np.abs(f - np.nanmedian(f, 1, keepdims=True)), 1),
            'min': np.nanmin(f, 1), 'max': np.nanmax(f, 1),
            'skew': stats.skew(f, 1, nan_policy='omit'), 'kurt': stats.kurtosis(f, 1, nan_policy='omit'),
            **{f'p{q}': np.nanpercentile(f, q, 1) for q in (1, 5, 25, 75, 95, 99)},
        }))
    X = pd.concat(out, ignore_index=True)
    return pd.concat([X, stars[['tmag', 'teff', 'logg', 'radius', 'contamination']].reset_index(drop=True)], axis=1)

X = simple_features(train_flux, train)
X_test = simple_features(test_flux, test)
X.head()

## Train and validate
Hold out 20% of the training stars to estimate the macro-F1 score. (The test set comes from a different sector, so expect your leaderboard score to be a bit different.)

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, train.label, test_size=0.2, stratify=train.label, random_state=42)
model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, class_weight='balanced', random_state=0)
model.fit(X_tr, y_tr)
print('validation macro-F1:', round(f1_score(y_va, model.predict(X_va), average='macro'), 4))

## Train on everything and write the submission

In [ ]:
model.fit(X, train.label)
sub = pd.DataFrame({'id': test.id, 'label': model.predict(X_test)})
sub.to_csv('submission.csv', index=False)
sub.label.value_counts()

## Ideas to climb the leaderboard
- **Detrend**: subtract the median curve of all stars on the same `camera`/`ccd`, then remove slow variations with a median filter.
- **Search for transits**: run a Box Least Squares search (`astropy.timeseries.BoxLeastSquares`) and fold on the best period.
- **Vet the dip**: depth, duration, U vs V shape, odd vs even depth, a secondary eclipse at phase 0.5, and **the centroid shift during the dip**.
- **Ignore or use the flags**: momentum dumps (`quality & 1`) cause blips that are not transits.
- **Frequencies**: a periodogram tells spots (days) from pulsations (hours).
- **Flares**: bright outliers with a decaying tail vs single-point cosmic rays.
- **Deep learning**: a 1D CNN on detrended flux + folded flux + centroids.